In [0]:
from pyspark.sql.functions import col,when,sum as s,lit
from pyspark.sql.types import IntegerType,FloatType,StringType

In [0]:
# Configure storage account key
spark.conf.set(
    "fs.azure.account.key.onlineaccstorage.dfs.core.windows.net",
    dbutils.secrets.get(scope="onlinesales", key="storagevalut")
)

storage  = "abfss://salesdata@onlineaccstorage.dfs.core.windows.net/bronze"
dbutils.fs.ls(storage)

In [0]:
accounts = spark.read.format("delta").load(f"{storage}/accounts")
display(accounts)

In [0]:
data_dictonary = spark.read.format("delta").load(f"{storage}/data_dictionary")
display(data_dictonary)

In [0]:
products = spark.read.format("delta").load(f"{storage}/product")
display(products)

In [0]:
sales_pipeline = spark.read.format("delta").load(f"{storage}/sales_pipeline")
display(sales_pipeline)

In [0]:
sales_team = spark.read.format("delta").load(f"{storage}/sales_teams")
display(sales_team)

In [0]:
accounts.printSchema()

In [0]:
accounts = accounts.withColumnRenamed("subsidiary_of","parent_company")
data_dictonary = data_dictonary.withColumnRenamed("Table","table")
data_dictonary = data_dictonary.withColumnRenamed("Field","field")
data_dictonary = data_dictonary.withColumnRenamed("Description","description")

In [0]:
accounts.display()

In [0]:
data_dictonary.display()

In [0]:
# Accounts
null_accounts = accounts.select(
    *[s(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in accounts.columns]
)
null_accounts.display()

# Products
null_products = products.select(
    *[s(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in products.columns]
)
null_products.display()

# Sales Pipeline
null_sales_pipeline = sales_pipeline.select(
    *[s(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in sales_pipeline.columns]
)
null_sales_pipeline.display()

# Sales Team
null_sales_team = sales_team.select(
    *[s(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in sales_team.columns]
)
null_sales_team.display()

# Data Dictionary
null_data_dictonary = data_dictonary.select(
    *[s(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in data_dictonary.columns]
)
null_data_dictonary.display()

In [0]:
accounts = accounts.fillna({"parent_company":"independet"})
sales_pipeline = sales_pipeline.fillna({"account":"unknown"})

In [0]:
null_accounts = accounts.select(
    [s(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in accounts.columns]
)
null_accounts.display()

In [0]:
null_sales_pipeline = sales_pipeline.select(
    [s(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in sales_pipeline.columns]
)
null_sales_pipeline.display()

In [0]:
sliver_storage = "abfss://salesdata@onlineaccstorage.dfs.core.windows.net"
sliver = sliver_storage + "/sliver"
accounts_store = sliver + "/accounts"
products_store = sliver + "/products"
sales_pipeline_store = sliver + "/sales_pipeline"
sales_team_store = sliver + "/sales_team"
data_dictonary_store = sliver + "/data_dictonary"

In [0]:
accounts.write.format("delta").mode("overwrite").option("mergeSchema",True).save(accounts_store)

In [0]:
data_dictonary.write.format("delta").mode("overwrite").option("mergeSchema",True).save(data_dictonary_store)
products.write.format("delta").mode("overwrite").option("mergeSchema",True).save(products_store)
sales_pipeline.write.format("delta").mode("overwrite").option("mergeSchema",True).save(sales_pipeline_store)
sales_team.write.format("delta").mode("overwrite").option("mergeSchema",True).save(sales_team_store)